In [ ]:
import os, sys, json, shutil, subprocess, importlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    from facenet_pytorch import MTCNN
except ImportError:
    _pip("--no-deps", "facenet-pytorch"); importlib.invalidate_caches()
    from facenet_pytorch import MTCNN
try:
    from hsemotion.facial_emotions import HSEmotionRecognizer
except ImportError:
    _pip("timm"); _pip("--no-deps", "hsemotion"); importlib.invalidate_caches()
    from hsemotion.facial_emotions import HSEmotionRecognizer
print("Libraries imported successfully.")

In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
with open(os.path.join(BASE_DIR, "config.json"), "r") as f:
    config = json.load(f)
torch.manual_seed(config["seed"]); np.random.seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR, FEATURES_DIR, RESULTS_DIR = config["paths"]["data_dir"], config["paths"]["features_dir"], config["paths"]["results_dir"]
SPLITS = ["train", "dev", "test"]
VIDEO_DIM = config["video_feature_dim"]
N_FRAMES, DECODE_FPS, CROP_SIZE = config["video_num_frames"], config["video_decode_fps"], config["video_frame_size"]
DET_W, DET_H = config.get("video_detect_size", [640, 360])
MAX_MISSING = config["max_missing_fraction"]                              
MAX_NO_FACE = config.get("video_max_missing_fraction", 0.30)             
MIN_FACE_PROB = config.get("video_min_face_prob", 0.90)
MIN_FACE_PX = config.get("video_min_face_px", 40)
HS_MODEL = config.get("video_hsemotion_model", "enet_b0_8_best_vgaf")
BACKBONE_TAG = f"mtcnn+hsemotion:{HS_MODEL}"
CROP_MARGIN = 0.15
CHUNK_VIDEOS, EMB_BATCH, CKPT_EVERY = 32, 64, 20
N_WORKERS = max(1, min(4, os.cpu_count() or 1))
STANDARDIZE = os.environ.get("MELD_P5_STANDARDIZE", "0").strip().lower() in ("1", "true", "yes")   
RECOMPUTE = os.environ.get("MELD_P5_RECOMPUTE", "0") == "1"

assert config.get("video_backbone") == "mtcnn+hsemotion", ("config.json still describes the old DenseNet setup -- re-run the PATCHED "
                                                           "Phase 1 notebook first (it only changes the video_* keys).")
manifests = {}
for s in SPLITS:
    m = pd.read_csv(os.path.join(DATA_DIR, f"media_manifest_{s}.csv"), keep_default_na=False)
    n_csv = len(pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv")))
    assert len(m) == n_csv and (m["row_idx"].values == np.arange(n_csv)).all(), f"[{s}] manifest/CSV misalignment"
    manifests[s] = m
    print(f"[{s}] manifest rows: {len(m)}  usable video clips: {(m['status'] == 'ok').sum()}")
print(f"Device: {device} | backbone: {BACKBONE_TAG} | dim {VIDEO_DIM} | {N_FRAMES} frames @ {DECODE_FPS} fps | detect {DET_W}x{DET_H} | "
      f"face prob>={MIN_FACE_PROB}, side>={MIN_FACE_PX}px | no-face limit {MAX_NO_FACE:.0%} | standardize={STANDARDIZE}")

In [ ]:
def best_face_box(boxes, probs, min_prob, min_px):
    """Largest accepted face. boxes: None or [n,4] (x1,y1,x2,y2); probs: None or [n]. Returns a length-4 float array or None."""
    if boxes is None or probs is None:
        return None
    best, best_area = None, 0.0
    for b, p in zip(np.asarray(boxes, dtype=np.float64).reshape(-1, 4), np.asarray(probs, dtype=np.float64).reshape(-1)):
        if not np.isfinite(b).all() or not np.isfinite(p) or p < min_prob:
            continue
        w, h = b[2] - b[0], b[3] - b[1]
        if min(w, h) < min_px:
            continue
        if w * h > best_area:
            best, best_area = b, w * h
    return best


def crop_with_margin(img, box, margin, out_size):
    """img uint8 [H,W,3]; returns uint8 [out_size,out_size,3] (square-ised box + margin, clipped to the image) or None."""
    H, W = img.shape[:2]
    cx, cy = (box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0
    half = max(box[2] - box[0], box[3] - box[1]) / 2.0 * (1.0 + 2.0 * margin)
    x1, y1, x2, y2 = int(max(0, round(cx - half))), int(max(0, round(cy - half))), int(min(W, round(cx + half))), int(min(H, round(cy + half)))
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return np.asarray(Image.fromarray(img[y1:y2, x1:x2]).resize((out_size, out_size), Image.BILINEAR))


def clip_feature(frame_feats):
    """frame_feats: [k,D] features of the frames that contain a face (k may be 0). Returns (vector or None, k)."""
    k = 0 if frame_feats is None else len(frame_feats)
    if k == 0:
        return None, 0
    v = np.asarray(frame_feats, dtype=np.float64).mean(axis=0).astype(np.float32)
    return (v, k) if np.isfinite(v).all() else (None, k)


def split_status(n, n_decode_fail, n_no_face, max_decode, max_no_face):
    """Abort rule. Decode / missing-file failures are bounded strictly; no-face clips (valid=False) by the looser limit."""
    f_dec, f_all = n_decode_fail / n, (n_decode_fail + n_no_face) / n
    if f_dec > max_decode:
        return False, f"decode/missing failures {f_dec:.2%} > strict limit {max_decode:.2%}"
    if f_all > max_no_face:
        return False, f"unusable clips (decode + no face) {f_all:.2%} > video_max_missing_fraction {max_no_face:.2%}"
    return True, "ok"


def zscore_fit(x, valid):
    mu = x[valid].mean(axis=0)
    sd = x[valid].std(axis=0)
    return mu.astype(np.float32), np.where(sd < 1e-6, 1.0, sd).astype(np.float32)
print("Helpers defined.")

In [ ]:
mtcnn = MTCNN(keep_all=True, min_face_size=int(MIN_FACE_PX), device=device, post_process=False)

def _load_hsemotion():
    dev = "gpu" if device.type == "cuda" else "cpu"
    try:
        return HSEmotionRecognizer(model_name=HS_MODEL, device=dev)
    except Exception as e:                                   
        print(f"HSEmotion load failed ({type(e).__name__}); retrying with torch.load(weights_only=False) for this trusted download.")
        orig = torch.load
        torch.load = lambda *a, **k: orig(*a, **{**k, "weights_only": False})
        try:
            return HSEmotionRecognizer(model_name=HS_MODEL, device=dev)
        finally:
            torch.load = orig
fer = _load_hsemotion()

import re
_OPTIONAL_NONE_ATTRS = {"conv_s2d", "bn_s2d"}

def repair_pickled_attrs(modules, probe, max_rounds=12):
    """Run probe(); on "'Cls' object has no attribute 'x'" with x a known optional slot, set x=None on every module of that class lacking it; retry."""
    fixed = []
    for _ in range(max_rounds):
        try:
            probe()
            return fixed
        except AttributeError as e:
            m = re.match(r"'(\w+)' object has no attribute '(\w+)'", str(e))
            if not m or m.group(2) not in _OPTIONAL_NONE_ATTRS:
                raise
            cls, attr = m.groups()
            hit = [mod for mod in modules() if type(mod).__name__ == cls and not hasattr(mod, attr)]
            if not hit:
                raise
            for mod in hit:
                setattr(mod, attr, None)
            fixed.append((cls, attr, len(hit)))
    raise RuntimeError(f"pickled-model repair did not converge: {fixed}")

def rebuild_from_state_dict(fer_obj):
    """Fallback: fresh model from the CURRENT timm, weights copied from the pickle. Strict about what may differ (classifier only)."""
    import timm
    old = fer_obj.model
    sd = old.state_dict()
    ck = [k for k in sd if k.endswith("classifier.weight")]
    new = timm.create_model("tf_efficientnet_b0", pretrained=False, num_classes=(sd[ck[0]].shape[0] if ck else 8))
    res = new.load_state_dict(sd, strict=False)
    bad = [k for k in list(res.missing_keys) + list(res.unexpected_keys) if not k.startswith("classifier")]
    assert not bad, f"state-dict rebuild would not match the pickled model: {bad[:6]}"
    if hasattr(old, "classifier"):
        new.classifier = old.classifier
    fer_obj.model = new.to(device).eval()
    print("Rebuilt the HSEmotion backbone from its state dict with the installed timm.")

_probe_img = [np.random.RandomState(1).randint(0, 255, (CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8)]
def _probe():
    with torch.no_grad():
        fer.extract_multi_features(_probe_img)
try:
    _fixed = repair_pickled_attrs(lambda: fer.model.modules(), _probe)
    if _fixed:
        print(f"Repaired pickled HSEmotion model for the installed timm: {_fixed}")
except Exception as _e:
    print(f"Attribute repair failed ({type(_e).__name__}: {str(_e)[:120]}); rebuilding the backbone from its state dict.")
    rebuild_from_state_dict(fer)
    _probe()

@torch.no_grad()
def embed_crops(crops):
    """list of uint8 RGB [CROP_SIZE,CROP_SIZE,3] -> float32 [len,D]."""
    out = []
    for i in range(0, len(crops), EMB_BATCH):
        part = crops[i:i + EMB_BATCH]
        f = np.asarray(fer.extract_multi_features(part), dtype=np.float32).reshape(len(part), -1)
        out.append(f)
    return np.concatenate(out) if out else np.zeros((0, VIDEO_DIM), np.float32)

rs = np.random.RandomState(0)
probe = [rs.randint(0, 255, (CROP_SIZE, CROP_SIZE, 3), dtype=np.uint8) for _ in range(3)]
e1, e2 = embed_crops(probe), embed_crops(probe)
assert e1.shape == (3, VIDEO_DIM), f"embedding dim {e1.shape[1]} != video_feature_dim {VIDEO_DIM}: set video_feature_dim={e1.shape[1]} in Phase 1 and re-run it"
assert np.isfinite(e1).all(), "non-finite embeddings"
assert np.abs(e1 - e2).max() < 1e-3, f"embedding is not deterministic (max diff {np.abs(e1 - e2).max():.2e}) -- model left in train mode?"
assert np.abs(e1[0] - e1[1]).mean() > 1e-4, "different inputs give identical embeddings"
b, p = mtcnn.detect([Image.fromarray(np.full((DET_H, DET_W, 3), 127, np.uint8))] * 2)       
print(f"Self-check OK: HSEmotion dim {e1.shape[1]}, deterministic; MTCNN batch detect returned {type(b).__name__} of len {len(b)}.")

In [ ]:
def decode_frames(video_path):
    """Returns (frames uint8 [N_FRAMES,DET_H,DET_W,3], None) or (None, reason)."""
    vf = (f"fps={DECODE_FPS},scale={DET_W}:{DET_H}:force_original_aspect_ratio=decrease,"
          f"pad={DET_W}:{DET_H}:(ow-iw)/2:(oh-ih)/2")
    cmd = ["ffmpeg", "-nostdin", "-v", "error", "-i", video_path, "-vf", vf, "-f", "rawvideo", "-pix_fmt", "rgb24", "pipe:1"]
    try:
        p = subprocess.run(cmd, capture_output=True, timeout=180)
    except subprocess.TimeoutExpired:
        return None, "ffmpeg_timeout"
    if p.returncode != 0:
        return None, "ffmpeg_failed: " + p.stderr.decode(errors="replace").strip().replace("\n", " ")[:200]
    fb = DET_W * DET_H * 3
    buf = np.frombuffer(p.stdout, dtype=np.uint8)
    t = buf.size // fb
    if t == 0:
        return None, "no_frames_decoded"
    frames = buf[: t * fb].reshape(t, DET_H, DET_W, 3)
    return frames[np.linspace(0, t - 1, N_FRAMES).round().astype(int)], None

@torch.no_grad()
def faces_for_clip(frames):
    """frames uint8 [N,H,W,3] -> list of 224x224 crops (largest accepted face per frame; frames without a face are skipped)."""
    boxes, probs = mtcnn.detect([Image.fromarray(f) for f in frames])
    crops = []
    for img, bx, pr in zip(frames, boxes, probs):
        box = best_face_box(bx, pr, MIN_FACE_PROB, MIN_FACE_PX)
        if box is not None:
            c = crop_with_margin(img, box, CROP_MARGIN, CROP_SIZE)
            if c is not None:
                crops.append(c)
    return crops
print("Decoding / face helpers defined.")

In [ ]:
META_PATH = os.path.join(FEATURES_DIR, "video_features_meta.json")
BACKUP_DIR = os.path.join(FEATURES_DIR, "_backup_densenet121")
META = {"backbone": BACKBONE_TAG, "dim": VIDEO_DIM, "n_frames": N_FRAMES, "decode_fps": DECODE_FPS, "detect_size": [DET_W, DET_H],
        "min_face_prob": MIN_FACE_PROB, "min_face_px": MIN_FACE_PX, "crop_margin": CROP_MARGIN}

def cache_ok():
    if RECOMPUTE or not os.path.exists(META_PATH):
        return False
    if json.load(open(META_PATH)) != META:
        return False
    return all(os.path.exists(os.path.join(FEATURES_DIR, f"video_features_raw_{s}.npy")) and
               os.path.exists(os.path.join(FEATURES_DIR, f"video_valid_{s}.npy")) for s in SPLITS)

def backup_legacy():
    """Move features produced by any other extractor out of the way (keeps them byte-for-byte)."""
    legacy = [f for s in SPLITS for f in (f"video_features_{s}.npy", f"video_valid_{s}.npy") if os.path.exists(os.path.join(FEATURES_DIR, f))]
    legacy += [f for s in SPLITS for f in (f"video_features_raw_{s}.npy",) if os.path.exists(os.path.join(FEATURES_DIR, f))]
    if not legacy:
        return
    os.makedirs(BACKUP_DIR, exist_ok=True)
    for f in legacy:
        dst = os.path.join(BACKUP_DIR, f)
        if os.path.exists(dst):
            dst = dst + ".dup"
        shutil.move(os.path.join(FEATURES_DIR, f), dst)
    for s in SPLITS:
        r = os.path.join(RESULTS_DIR, f"video_extraction_report_{s}.json")
        if os.path.exists(r) and not os.path.exists(os.path.join(BACKUP_DIR, os.path.basename(r))):
            shutil.copy2(r, os.path.join(BACKUP_DIR, os.path.basename(r)))
    print(f"Moved {len(legacy)} legacy video cache files to {BACKUP_DIR}")

def process_split(split, executor):
    raw_path = os.path.join(FEATURES_DIR, f"video_features_raw_{split}.npy")
    valid_path = os.path.join(FEATURES_DIR, f"video_valid_{split}.npy")
    nface_path = os.path.join(FEATURES_DIR, f"video_face_frames_{split}.npy")
    part_path = os.path.join(FEATURES_DIR, f"video_partial_{split}.npz")
    if os.path.exists(raw_path) and os.path.exists(valid_path) and not RECOMPUTE:     
        print(f"[{split}] raw features already complete -> skipping.")
        return
    m = manifests[split]; n = len(m)
    reasons = {int(i): f"video_{st}" for i, st in zip(m["row_idx"], m["status"]) if st != "ok"}
    feats = np.zeros((n, VIDEO_DIM), dtype=np.float32); valid = np.zeros(n, dtype=bool); nface = np.zeros(n, dtype=np.int16)
    ok = m[m["status"] == "ok"]; rows, paths = ok["row_idx"].tolist(), ok["video_path"].tolist()
    start0 = 0
    if os.path.exists(part_path) and not RECOMPUTE:
        z = np.load(part_path, allow_pickle=False)
        if json.loads(str(z["meta"])) == META:
            feats, valid, nface, start0 = z["feats"], z["valid"], z["nface"], int(z["next_start"])
            reasons.update({int(k): v for k, v in json.loads(str(z["reasons"])).items()})
            print(f"[{split}] resuming at clip {start0}/{len(rows)}")

    def save_partial(next_start):
        tmp = part_path + ".tmp.npz"
        np.savez(tmp, feats=feats, valid=valid, nface=nface, next_start=next_start, meta=json.dumps(META),
                 reasons=json.dumps({str(k): v for k, v in reasons.items()}))
        os.replace(tmp, part_path)

    for ci, start in enumerate(tqdm(range(start0, len(rows), CHUNK_VIDEOS), desc=f"faces+HSEmotion ({split})", initial=start0 // CHUNK_VIDEOS,
                                     total=(len(rows) + CHUNK_VIDEOS - 1) // CHUNK_VIDEOS)):
        chunk_rows, chunk_paths = rows[start:start + CHUNK_VIDEOS], paths[start:start + CHUNK_VIDEOS]
        decoded = list(executor.map(decode_frames, chunk_paths))
        all_crops, owners = [], []
        for r, (frames, err) in zip(chunk_rows, decoded):
            if err is not None:
                reasons[int(r)] = err
                continue
            crops = faces_for_clip(frames)
            nface[r] = len(crops)
            if not crops:
                reasons[int(r)] = "no_face"
                continue
            all_crops.extend(crops); owners.extend([r] * len(crops))
        if all_crops:
            emb, owners_a = embed_crops(all_crops), np.asarray(owners)
            for r in sorted(set(owners)):
                v, k = clip_feature(emb[owners_a == r])
                if v is None:
                    reasons[int(r)] = "non_finite_features"
                else:
                    feats[r], valid[r] = v, True
        if (ci + 1) % CKPT_EVERY == 0:
            save_partial(start + CHUNK_VIDEOS)

    failures = [{"row_idx": i, "Dialogue_ID": int(m.at[i, "Dialogue_ID"]), "Utterance_ID": int(m.at[i, "Utterance_ID"]), "reason": r}
                for i, r in sorted(reasons.items())]
    n_no_face = sum(1 for r in reasons.values() if r == "no_face")
    n_decode = len(failures) - n_no_face
    hist = np.bincount(nface.clip(0, N_FRAMES), minlength=N_FRAMES + 1).tolist()
    with open(os.path.join(RESULTS_DIR, f"video_extraction_report_{split}.json"), "w") as f:
        json.dump({"split": split, "backbone": BACKBONE_TAG, "n": n, "n_valid": int(valid.sum()), "n_no_face": n_no_face,
                   "n_decode_or_missing": n_decode, "fraction_no_face": n_no_face / n, "fraction_decode_or_missing": n_decode / n,
                   "frames_with_face_histogram_0_to_N": hist, "failures": failures}, f, indent=2)
    print(f"[{split}] valid {int(valid.sum())}/{n} | no-face {n_no_face} ({n_no_face / n:.2%}) | decode/missing {n_decode} ({n_decode / n:.2%}) | "
          f"frames-with-face histogram {hist}")
    good, why = split_status(n, n_decode, n_no_face, MAX_MISSING, MAX_NO_FACE)
    if not good:
        raise RuntimeError(f"[{split}] {why}. See results/video_extraction_report_{split}.json (partial progress kept in {part_path})")
    np.save(raw_path, feats); np.save(valid_path, valid); np.save(nface_path, nface)
    if os.path.exists(part_path):
        os.remove(part_path)
    print(f"Saved raw video features for {split}: {feats.shape}")

if cache_ok():
    print("Face features already cached with identical settings -> skipping extraction.")
else:
    prev_meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else None
    if prev_meta != META and not RECOMPUTE:
        backup_legacy()                                 
    with open(META_PATH, "w") as f:                      
        json.dump(META, f, indent=2)
    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        for s in SPLITS:
            process_split(s, executor)

In [ ]:
raw = {s: np.load(os.path.join(FEATURES_DIR, f"video_features_raw_{s}.npy")) for s in SPLITS}
vmask = {s: np.load(os.path.join(FEATURES_DIR, f"video_valid_{s}.npy")) for s in SPLITS}
if STANDARDIZE:
    mu, sd = zscore_fit(raw["train"], vmask["train"])
    np.savez(os.path.join(FEATURES_DIR, "video_scaler.npz"), mean=mu, std=sd, n_fit=int(vmask["train"].sum()))
for s in SPLITS:
    x = raw[s].copy()
    if STANDARDIZE:
        x = ((x - mu) / sd).astype(np.float32)
        x[~vmask[s]] = 0.0
    np.save(os.path.join(FEATURES_DIR, f"video_features_{s}.npy"), x)

for s in SPLITS:
    x = np.load(os.path.join(FEATURES_DIR, f"video_features_{s}.npy")); v = vmask[s]
    n_csv = len(pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv")))
    assert x.shape == (n_csv, VIDEO_DIM), f"[{s}] bad shape {x.shape}"
    assert np.isfinite(x).all(), f"[{s}] non-finite values"
    assert (x[~v] == 0).all(), f"[{s}] unusable rows must be all-zero"
    assert x[v].std(axis=0).mean() > 1e-4, f"[{s}] video features are (near-)constant -- extraction is not working"
    assert (np.abs(x[v]).sum(axis=1) > 0).all(), f"[{s}] valid rows containing all-zero features"
    print(f"[{s}] OK: {int(v.sum())}/{n_csv} valid ({v.mean():.1%}); mean per-dim std {x[v].std(axis=0).mean():.4f}")
print("Phase 5 Video Feature Extraction (faces) completed. Next: Step 2 (audio) -- do NOT start Phase 6+ yet; Phase 6 still hard-codes the old dims.")